In [3]:
import duckdb
import matplotlib.pyplot as plt

con = duckdb.connect()
con.sql("""
    CREATE VIEW v AS
    SELECT * FROM read_parquet('../data/clean/*/*/violations.parquet')
""")

CountBIGINT


In [4]:
con.sql("""
    CREATE OR REPLACE VIEW t AS
    SELECT *, CAST(issued_at_utc AT TIME ZONE 'America/New_York' AS DATE) AS local_date
    FROM v WHERE is_geocoded
""")

train_weeks = (date(2026, 1, 1) - date(2023, 1, 1)).days / 7   # subtract excluded weeks if any

con.sql(f"""
    CREATE OR REPLACE TABLE baseline AS
    SELECT latitude, longitude,
           issued_local_weekday AS weekday, issued_local_hour AS hour,
           count(*) / {train_weeks} AS tickets_per_week
    FROM t WHERE local_date < DATE '2026-01-01'
    GROUP BY ALL
""")

NameError: name 'date' is not defined